In [ ]:
using Plots
using LinearAlgebra
using BenchmarkTools
using Revise
using Jun4Project
using Krylov
using DifferentialEquations
using FFTW
using Printf
using SparseArrays

# Heat in 2D

In [ ]:
n = 100;
nx = n;
ny = n;
Nx = nx-1;
Ny = ny-1;
Lx = 5;
Ly = 5;
x = LinRange(-Lx, Lx, nx + 1)
y = LinRange(-Ly, Ly, ny + 1)
Δx = x[2] - x[1];
Δy = y[2] - y[1];
xy = [[x_,y_] for x_ in x, y_ in y]
xy_int = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);


In [ ]:
f(X,t) = exp(-2 *(X[1]^2 + X[2]^2)) * sin(2π*t)^2; #spatially localized, time dependent.
# f(X,t) = (cos(X[1])*sin(X[2]))^2 * sin(2π*t)^2
F(X,t) = vec(f.(X, t))

In [ ]:
# U and u share the same memory
u = zeros(Nx,Ny);
U = vec(u);
u_vals = [deepcopy(u)];
t_vals = LinRange(0, 25, 251);
Δt = t_vals[2] - t_vals[1];
t = t_vals[1];
# f2d and Fvec share the same memory
f2d = zeros(Nx,Ny);
Fvec = vec(f2d);
for t in t_vals[2:end]
    # .= copies over the values
    f2d .= f.(xy_int, t);
    # .= copies over the values
    U .= cg(I-Δt*L, U + Δt*Fvec)[1];
    push!(u_vals, deepcopy(u));
end    

In [ ]:
anim = @animate for (t,u) in zip(t_vals[1:1:end], u_vals[1:1:end])
    contourf(x[2:end-1], y[2:end-1], u', title = @sprintf("t = %.2f", t), xlabel="x", ylabel="y", c=:viridis, 
        clims = (0, 1), colorbar=true)
    xlims!(-Lx, Lx)
    ylims!(-Ly, Ly)

end
gif(anim, fps = 10)

# Schrodinger

In [ ]:
n = 100;
nx = n;
ny = n;
Nx = nx-1;
Ny = ny-1;
Lx = 5;
Ly = 5;
x = LinRange(-Lx, Lx, nx + 1)
y = LinRange(-Ly, Ly, ny + 1)
Δx = x[2] - x[1];
Δy = y[2] - y[1];
xy = [[x_,y_] for x_ in x, y_ in y]
xy_int = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);


In [ ]:
# V(X) = 0.5 * X⋅X; # harmonic potential, ||x||^2/2
V(X) = cos(2*pi*X[1]) * cos(2*pi*X[2]); # periodic potential
Vvec = vec(V.(xy_int));
Vmat = spdiagm(Vvec)

In [ ]:
u = Complex.(ones(Nx,Ny));
U = vec(u);
u_vals = [deepcopy(u)];
t_vals = LinRange(0, 25, 251);
Δt = t_vals[2] - t_vals[1];

for t in t_vals[2:end]
    # U .= cg(im*I + 0.5 * Δt*L - 0.5 * Δt * Vmat,(im*I - 0.5 * Δt*L + 0.5 * Δt * Vmat)*U )[1]; 
    U .= (im*I + 0.5 * Δt*L - 0.5 * Δt * Vmat)\((im*I - 0.5 * Δt*L + 0.5 * Δt * Vmat)*U);
    push!(u_vals, deepcopy(u));
end    

In [ ]:
anim = @animate for (t,u) in zip(t_vals[1:1:end], u_vals[1:1:end])
    contourf(x[2:end-1], y[2:end-1], real.(u)', title = @sprintf("t = %.2f", t), xlabel="x", ylabel="y", c=:viridis, 
        clims = (-2, 2), colorbar=true)
    xlims!(-Lx, Lx)
    ylims!(-Ly, Ly)

end
gif(anim, fps = 6)

In [ ]:
anim = @animate for (t,u) in zip(t_vals[1:5:end], u_vals[1:5:end])
    contourf(x[2:end-1], y[2:end-1], (abs.(u).^2)', title = @sprintf("t = %.2f", t), xlabel="x", ylabel="y", c=:viridis, 
        clims = (0, 2), colorbar=true)
    xlims!(-Lx, Lx)
    ylims!(-Ly, Ly)

end
gif(anim, fps = 6)

# KdV

In [ ]:

# KdV: u_t + u u_x + u_xxx = 0 on a periodic domain x in [0, L)
# These defaults are chosen to run quickly in a notebook; increase N/tspan for higher fidelity.
N = 512
L = 4π
x = L .* (0:N-1) ./ N

# Fourier wave numbers
k = (2π / L) .* vcat(0:N÷2, -N÷2+1:-1)
ik = 1im .* k
# ik3 = ik .^ 3
ik3 = 1im .* k.^3

function kdv_if_pseudospectral!(dvhat, vhat, p, t)
    ik, ik3 = p

    Eminus = @. exp(-ik3 * t);
    Eplus = @. exp(ik3 * t);
    uhat = @. Eplus * vhat;

    u = real(ifft(uhat));

    nlhat = fft(u.^2);
    # nlhat[dealias] .= 0
    @. dvhat = -Eminus * ik * 0.5 * nlhat;
    dvhat
end


In [ ]:

# Periodic initial condition
# u0 = @. exp(-4 * (x - L/2)^2)
A = 25; B = 16;
u0 = @. 3 * A^2 * sech(.5 *(A* (x - L/2+2)))^2 +  3 * B^2 * sech(.5 *(B *(x - L/2+1)))^2 
vhat0 = fft(u0)

tspan = (0.0, 0.01)
# p = (ik, ik3, dealias)
p = (ik, ik3)

prob = ODEProblem(kdv_if_pseudospectral!, vhat0, tspan, p)

# DifferentialEquations.jl is used for time stepping.
sol = solve(prob; saveat=0.0001)

In [ ]:

# Reconstruct u(x,t) from integrating-factor variable v̂
Ucols = [real.(ifft(exp.(ik3 .* t) .* vhat)) for (t, vhat) in zip(sol.t, sol.u)]
U = reduce(hcat, Ucols)  # size (N, nt)

contourf(
    x, sol.t, U',
    xlabel="x", ylabel="t",
    title="KdV solution via pseudospectral method",
    c=:viridis, colorbar=true
    )

In [ ]:
anim = @animate for i in 1:length(sol.t)
    plot(x, U[:, i], ylim=(-250, 2500), 
    title="t = $(round(sol.t[i], digits=6))", label="")
    xlabel!("x")
    ylabel!("u")
end
gif(anim, fps=6)